In [1]:
from pathlib import Path
import xmltodict
import pandas as pd
import numpy as np
import tifffile as tiff
import xmltodict
from dateutil import parser
import re

import matplotlib.pyplot as plt
from collections import namedtuple
from shapely.geometry import box

from skimage.registration import phase_cross_correlation
from skimage.transform import rescale, resize, downscale_local_mean
from skimage import exposure

import zarr
from pylibCZIrw import czi as pyczi

In [2]:
"""Skip during test
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Intermediate")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Proximal")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Distal")
section_folders = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_dir() and folder.name.startswith("S_"):
        print(f"fould series folder: {folder.name}")
        section_folders.append(folder)
"""

'Skip during test\nseries_folder = Path(r"E:\\PROJECTS\\EM\\LUKE\\TA23\\Intermediate")\nseries_folder = Path(r"E:\\PROJECTS\\EM\\LUKE\\TA23\\Proximal")\nseries_folder = Path(r"E:\\PROJECTS\\EM\\LUKE\\TA23\\Distal")\nsection_folders = []\nfor folder in series_folder.iterdir():  # Iterate over all items in the folder\n    if folder.is_dir() and folder.name.startswith("S_"):\n        print(f"fould series folder: {folder.name}")\n        section_folders.append(folder)\n'

In [3]:
from atlas.io import is_there_a_single_tif, extract_s_number

In [4]:
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS-projects\L32-10-2-w2-ROI-20nm-BSD")
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS-projects\L32-10-1-ROI-w1-20nm-bsd")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\ATLAS-projects\TA31-atlas\Stitched-Sets\TA31-20nm-roi1-2")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\ATLAS-projects\TA31-atlas\Stitched-Sets\TA31-20nm-roi3")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA29\test")

# Find the first CSV file that starts with "PhaseCC_stitching_S_"
csv_file = next(series_folder.glob("PhaseCC_stitching_S_*.csv"), None)

if csv_file is None:
    raise FileNotFoundError("No matching CSV file found in the folder.")

print(f"Found CSV file: {csv_file.name}")

# Load the CSV into a DataFrame
df = pd.read_csv(csv_file)

# Try to find a 'pixelSize' column or entry
if 'PixelSizeMicron' in df.columns:
    pix_size_micron =  df['PixelSizeMicron'].max()
else:
    raise ValueError("No 'pixelSize' parameter found in the CSV.")

print(f"pixel_size in microns {pix_size_micron}")


#TODO: important change this to metadata readout
pixel_size = {
    'Value': pix_size_micron,
    'Axial': 0.070,
    'Unit': 'µm'
}
print("work on pixel size")
""" EXAMPLE
with open(tif_folder.joinpath('tif-stack-metadata.xml'), "rb") as f:
    metadata_dict = xmltodict.parse(f, xml_attribs=True)

pixel_size = metadata_dict['human_meta_data']['pixel_size']
"""

tif_list = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_file() and folder.name.endswith(".tiff"):
        print(f"fould series image: {folder.name}")
        tif_list.append(folder)

Found CSV file: phaseCC_stitching_S_80.csv
pixel_size in microns 0.0099999696429462
work on pixel size
fould series image: stitched_image_S_80.tiff
fould series image: stitched_image_S_81.tiff


In [5]:
"""SKIP during test
tif_list = []
for section_folder in section_folders:
    single_tif, tif_path = is_there_a_single_tif(section_folder)
    if single_tif:
        #print(tif_path)
        tif_list.append(tif_path)
"""

tif_list_sorted = sorted(tif_list, key=extract_s_number)
from atlas.io import reorder_files_by_s_number, parse_shorthand_order

In [6]:
correct_order_path = Path(series_folder.joinpath("correct_order.txt"))

try:
    new_order = parse_shorthand_order(correct_order_path)
    print(f"Loaded correct_order.txt: {new_order}")

    tif_list_sorted = reorder_files_by_s_number(tif_list_sorted, new_order)
    print("Files reordered based on correct_order.txt:")
    for f in tif_list_sorted:
        print(f)

except FileNotFoundError:
    print("No correct_order.txt found — using original order.")
except Exception as e:
    print(f"Error reading or applying correct_order.txt: {e}")

No correct_order.txt found — using original order.


In [ ]:
from atlas.io.fibics_metadata import extract_tif_metadata, get_pixel_size_from_tif
from atlas.io import create_empty_folder, rm_tree, apply_alignment, zarr_array_to_czi
from atlas.image_analysis import image_dtype_min_max, mask_low_and_saturation, rescale_image_intensity
from atlas.alignment import calculate_cumulative_shifts, initialize_alignment_df, pairwise_alignment, first_last_true, ROI


In [12]:
output_path = series_folder.joinpath("alignment_results")
create_empty_folder(output_path)

down_scale = 10

# initialize the dataframe, in particular we asign the pair-wise patching of the images
# based on the sorted list of tiff.
z_align_df = initialize_alignment_df(tif_list_sorted, down_scale)
# run pairwise alignment based on the provided dataframe
z_align_df = pairwise_alignment(z_align_df)
# now we can calculate the cumulative shifts
z_align_df = calculate_cumulative_shifts(z_align_df)

# ✅ At the end, save the DataFrame as a CSV for later analysis
z_align_df_path = output_path.joinpath("z_alignment_results.pkl")
z_align_df.to_pickle(z_align_df_path)

z_align_df 

Created new (or emptied) folder: E:\PROJECTS\EM\LUKE\TA29\test\alignment_results
Processing alignment: ref -> stitched_image_S_80.tiff, moving -> stitched_image_S_80.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [0. 0.]
current shift: [0. 0.]
Processing alignment: ref -> stitched_image_S_80.tiff, moving -> stitched_image_S_81.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [-64.   2.]
current shift: [-64.   2.]


,moving_path,reference_path,moving_ROI,reference_ROI,current_shift,cumulative_ROI,cumulative_shift,down_scale
0,E:\PROJECTS\EM\LUKE\TA29\test\stitched_image_S...,E:\PROJECTS\EM\LUKE\TA29\test\stitched_image_S...,"(0, 584, 0, 954)","(0, 584, 0, 954)","[0.0, 0.0]","(0, 584, 0, 954)","[0.0, 0.0]",10
1,E:\PROJECTS\EM\LUKE\TA29\test\stitched_image_S...,E:\PROJECTS\EM\LUKE\TA29\test\stitched_image_S...,"(0, 596, 0, 1055)","(0, 584, 0, 954)","[-64.0, 2.0]","(2, 598, -64, 991)","[-64.0, 2.0]",10


In [ ]:
z_align_df = pd.read_pickle(z_align_df_path)
z_align_df

,moving_path,reference_path,moving_ROI,reference_ROI,current_shift,cumulative_ROI,cumulative_shift,down_scale
0,E:\PROJECTS\EM\LUKE\TA29\test\stitched_image_S...,E:\PROJECTS\EM\LUKE\TA29\test\stitched_image_S...,"(0, 584, 0, 954)","(0, 584, 0, 954)","[0.0, 0.0]","(0, 584, 0, 954)","[0.0, 0.0]",10
1,E:\PROJECTS\EM\LUKE\TA29\test\stitched_image_S...,E:\PROJECTS\EM\LUKE\TA29\test\stitched_image_S...,"(0, 596, 0, 1055)","(0, 584, 0, 954)","[-64.0, 2.0]","(2, 598, -64, 991)","[-64.0, 2.0]",10


In [14]:
use_down_sample = True
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, use_down_sample=use_down_sample)

out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

we will downsample during saving of the zarr, this is good during testing for visual inspection
shape of the zarr array to create: (1095, 638, 2)
creating zarr at: E:\PROJECTS\EM\LUKE\TA29\test\test.zarr
Done for idx 0
Done for idx 1
Saving CZI file to: E:\PROJECTS\EM\LUKE\TA29\test\test_ds_aligned.czi
Processing frame 0
Processing frame 1


In [ ]:
if zarr_path.exists():
  rm_tree(zarr_path)

In [ ]:
use_down_sample = False
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, use_down_sample=use_down_sample)

In [ ]:
out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

In [ ]:
raise excit

In [ ]:
from atlas.alignment.utils import get_translation, first_last_true, ROI
from scipy.ndimage import rotate
from scipy.ndimage import shift as nd_shift

def get_translation(reference_img, moving_img, angle_range=(-2, 2), angle_steps=21):
    """
    Computes the best rotation and translation to align moving_img to reference_img using
    a sweep of small rotation angles and phase cross-correlation.

    Parameters:
    ----------
    reference_img : np.ndarray
        The reference image.
    moving_img : np.ndarray
        The moving image to be aligned.
    angle_range : tuple
        Min and max rotation angles in degrees (e.g., (-2, 2)).
    angle_steps : int
        Number of angles to try between angle_range[0] and angle_range[1].

    Returns:
    -------
    ROI
        The ROI used from the reference image.
    ROI
        The ROI used from the moving image.
    np.ndarray
        The final pixel shift (row, col) needed to align the moving image.
    float
        The best rotation angle in degrees.
    """
    assert isinstance(reference_img, np.ndarray)
    assert isinstance(moving_img, np.ndarray)

    mask_ref = np.logical_not(mask_low_and_saturation(reference_img))

    # Compute ROI from reference image
    x0, x1 = first_last_true(np.any(mask_ref, axis=0))
    y0, y1 = first_last_true(np.any(mask_ref, axis=1))
    ref_ROI = ROI(x0=x0, x1=x1, y0=y0, y1=y1)
    crop_ref_img  = reference_img[ref_ROI.y0:ref_ROI.y1, ref_ROI.x0:ref_ROI.x1]
    crop_ref_mask = mask_ref[ref_ROI.y0:ref_ROI.y1, ref_ROI.x0:ref_ROI.x1]

    # Sweep through angles
    best_error = np.inf
    best_angle = 0
    best_shift = None
    best_mov_ROI = None

    angles = np.linspace(angle_range[0], angle_range[1], angle_steps)

    for angle in angles:
        rotated_mov = rotate(moving_img, angle, reshape=False, order=1)
        mov_mask = np.logical_not(mask_low_and_saturation(rotated_mov))

        # Find ROI from rotated image
        x0, x1 = first_last_true(np.any(mov_mask, axis=0))
        y0, y1 = first_last_true(np.any(mov_mask, axis=1))
        mov_ROI = ROI(x0=x0, x1=x1, y0=y0, y1=y1)

        crop_mov_img = rotated_mov[mov_ROI.y0:mov_ROI.y1, mov_ROI.x0:mov_ROI.x1]
        crop_mov_mask = mov_mask[mov_ROI.y0:mov_ROI.y1, mov_ROI.x0:mov_ROI.x1]

        crop_shift = np.array([ref_ROI.y0 - mov_ROI.y0, ref_ROI.x0 - mov_ROI.x0])

        try:
            
            shift, error, _ = phase_cross_correlation(
                crop_ref_img, crop_mov_img,
                reference_mask=crop_ref_mask,
                moving_mask=crop_mov_mask,
                upsample_factor=1
            )
            shifted_img = nd_shift(crop_mov_img, shift=shift, order=1, mode='constant', cval=0.0)
            print(f"ref: {crop_ref_img.shape}, mov: {crop_mov_img.shape}, shifted: {shifted_img.shape} ")
            #mask_1 = np.logical_not(mask_low_and_saturation(shifted_img))
            #mask_2 = np.logical_not(mask_low_and_saturation(crop_ref_img))
            #total_mask = np.logical_and(mask_1, mask_2)
            #residual = crop_ref_img - shifted_img
            #residual = residual[total_mask]
            #error = np.mean(np.sum(np.abs(residual)))

        except Exception as e:
            print(f"Skipping angle {angle} due to error: {e}")
            continue

        total_shift = np.round(shift) + crop_shift

        print(f"angle: {angle}, shift: {shift}, total_shift {total_shift}, error {error}")

        if error < best_error:
            best_error = error
            best_angle = angle
            best_shift = total_shift
            best_mov_ROI = mov_ROI

    print(f"Best angle: {best_angle:.3f}°")
    print(f"Total shift: {best_shift}")

    return ref_ROI, best_mov_ROI, best_shift, best_angle

In [ ]:
from skimage.registration import optical_flow_tvl1, optical_flow_ilk
from skimage.transform import warp

In [ ]:
# Load moving image
mov_path = z_align_df["moving_path"][7]
mov_img = tiff.imread(mov_path)

# Load reference image
ref_path = z_align_df["reference_path"][7]
ref_img = tiff.imread(ref_path)

# ✅ Downscale for performance optimization
mov_img_ds = downscale_local_mean(mov_img, (down_scale, down_scale)).astype(mov_img.dtype)
ref_img_ds = downscale_local_mean(ref_img, (down_scale, down_scale)).astype(ref_img.dtype)

In [ ]:
print(f"Processing alignment: ref -> {ref_path.name}, moving -> {mov_path.name}")

# ✅ Compute translation shift
# --- Compute the optical flow
v, u = optical_flow_tvl1(reference_image=ref_img_ds, moving_image=mov_img_ds, num_warp=1, attachment=10, tightness=1.)

# --- Use the estimated optical flow for registration

nr, nc = ref_img_ds.shape

row_coords, col_coords = np.meshgrid(np.arange(nr), np.arange(nc), indexing='ij')

image1_warp = warp(mov_img_ds, np.array([row_coords + v, col_coords + u]), mode='edge', preserve_range=True)

image1_warp = image1_warp.astype(mov_img_ds.dtype)


In [ ]:
newczi = Path("test6.czi")

with pyczi.create_czi(newczi, exist_ok=True) as czidoc_w:
    # Loop over Z-planes and channels
    plane = {'C': 0, 'Z': 0, 'T': 0}
    czidoc_w.write(data=image_reference[..., np.newaxis], plane=plane)
    plane = {'C': 0, 'Z': 1, 'T': 0}
    czidoc_w.write(data=aligned[..., np.newaxis], plane=plane)
    plane = {'C': 0, 'Z': 2, 'T': 0}
    czidoc_w.write(data=image_moving[..., np.newaxis], plane=plane)

In [ ]:
# Load moving image
mov_path = Path(r"E:\PROJECTS\EM\LUKE\ATLAS-projects\TA31-atlas\TA31-luke_data\session_862126381\Section Set 3\stitched_image_S_9.tiff")
mov_img = tiff.imread(mov_path)

# Load reference image
ref_path = Path(r"E:\PROJECTS\EM\LUKE\ATLAS-projects\TA31-atlas\TA31-luke_data\session_862126381\Section Set 3\stitched_image_S_8.tiff")
ref_img = tiff.imread(ref_path)

# ✅ Downscale for performance optimization
mov_img_ds = downscale_local_mean(mov_img, (down_scale, down_scale)).astype(mov_img.dtype)
ref_img_ds = downscale_local_mean(ref_img, (down_scale, down_scale)).astype(ref_img.dtype)

# Create 1 row, 2 columns
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot first image
axes[0].imshow(mov_img_ds, cmap='gray')
axes[0].set_title("mov image")
axes[0].axis('off')

axes[1].imshow(ref_img_ds, cmap='gray')
axes[1].set_title("fix iamge")
axes[1].axis('off')

In [ ]:
import numpy as np
from scipy.fft import fft2, fftshift
from skimage.transform import warp_polar, rotate, estimate_transform, warp
from skimage.registration import phase_cross_correlation
from scipy.ndimage import shift as nd_shift

def crop_to_center_overlap(img1, img2):
    """
    Crops two images to the largest common size, centered on their respective centers.

    Parameters:
    -----------
    img1, img2 : np.ndarray
        Input images of possibly different shapes.

    Returns:
    --------
    cropped1, cropped2 : np.ndarray
        Cropped images of the same shape, center-aligned.
    """
    shape1 = np.array(img1.shape)
    shape2 = np.array(img2.shape)

    # Determine minimum shape in each dimension
    min_shape = np.minimum(shape1, shape2)

    # Compute center points
    center1 = shape1 // 2
    center2 = shape2 // 2

    # Compute start and end indices for cropping
    start1 = center1 - min_shape // 2
    end1   = start1 + min_shape

    start2 = center2 - min_shape // 2
    end2   = start2 + min_shape

    cropped1 = img1[start1[0]:end1[0], start1[1]:end1[1]]
    cropped2 = img2[start2[0]:end2[0], start2[1]:end2[1]]

    return cropped1, cropped2

def estimate_rotation_translation_fourier(reference_img, moving_img, upsample_factor=10):
    assert reference_img.shape == moving_img.shape, "Images must be the same shape"

    # Store original dtype
    orig_dtype = moving_img.dtype

    # Step 1: FFT magnitude
    f_ref = fftshift(fft2(reference_img))
    f_mov = fftshift(fft2(moving_img))
    mag_ref = np.abs(f_ref)
    mag_mov = np.abs(f_mov)

    # Step 2: Polar transform (linear for rotation only)
    center = np.array(reference_img.shape) / 2
    polar_ref = warp_polar(mag_ref, center=center, scaling='linear', preserve_range=True)
    polar_mov = warp_polar(mag_mov, center=center, scaling='linear', preserve_range=True)

    # Step 3: Estimate rotation angle
    shift, _, _ = phase_cross_correlation(polar_ref, polar_mov, upsample_factor=upsample_factor)
    rotation_deg = -shift[0] * 360 / polar_ref.shape[0]

    # Step 4: Apply rotation (preserve original data range)
    rotated = rotate(moving_img, rotation_deg, resize=False, order=2, mode='constant', cval=0.0, preserve_range=True)

    # Step 5: Estimate translation
    translation_shift, _, _ = phase_cross_correlation(reference_img, rotated, upsample_factor=upsample_factor)

    # Step 6: Apply translation (preserve range by keeping dtype conversion to the end)
    aligned = nd_shift(rotated, shift=translation_shift, order=2, mode='constant', cval=0.0)

    # Step 7: Cast back to original dtype
    aligned_final = aligned.astype(orig_dtype)

    return rotation_deg, translation_shift, aligned_final

def find_largest_centered_rectangle(mask):
    """
    Finds the largest rectangle centered at the image center
    that is fully contained in a binary mask.
    """
    h, w = mask.shape
    cy, cx = h // 2, w // 2

    max_up = max_down = max_left = max_right = 0

    # Vertical extent
    for i in range(cy, -1, -1):
        if mask[i, cx]:
            max_up += 1
        else:
            break
    for i in range(cy + 1, h):
        if mask[i, cx]:
            max_down += 1
        else:
            break

    # Horizontal extent
    for j in range(cx, -1, -1):
        if mask[cy, j]:
            max_left += 1
        else:
            break
    for j in range(cx + 1, w):
        if mask[cy, j]:
            max_right += 1
        else:
            break

    top = cy - max_up + 1
    bottom = cy + max_down
    left = cx - max_left + 1
    right = cx + max_right

    # Now shrink until the whole region is valid
    while True:
        region = mask[top:bottom, left:right]
        if region.shape[0] == 0 or region.shape[1] == 0:
            raise ValueError("No valid center-aligned rectangle found.")
        if np.all(region):
            break
        # shrink evenly
        if bottom - top > 1:
            top += 1
            bottom -= 1
        if right - left > 1:
            left += 1
            right -= 1

    return top, bottom, left, right

def crop_to_centered_valid_rectangle(img1, img2):
    shape = np.minimum(img1.shape, img2.shape)
    y, x = shape

    # Compute masks
    mask1 = np.logical_not(mask_low_and_saturation(img1)[:y, :x])
    mask2 = np.logical_not(mask_low_and_saturation(img2)[:y, :x])

    # Combine masks
    common_mask = mask1 & mask2

    # Find largest internal rectangle centered on center
    top, bottom, left, right = find_largest_centered_rectangle(common_mask)

    # Apply to both images
    center1 = np.array(img1.shape) // 2
    center2 = np.array(img2.shape) // 2

    crop1 = img1[
        center1[0] - (y//2 - top):center1[0] + (bottom - y//2),
        center1[1] - (x//2 - left):center1[1] + (right - x//2)
    ]
    crop2 = img2[
        center2[0] - (y//2 - top):center2[0] + (bottom - y//2),
        center2[1] - (x//2 - left):center2[1] + (right - x//2)
    ]

    return crop1, crop2


In [ ]:
image_reference, image_moving = crop_to_centered_valid_rectangle(ref_img_ds, mov_img_ds)
# Create 1 row, 2 columns
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot first image
axes[0].imshow(image_reference, cmap='gray')
axes[0].set_title("fix image")
axes[0].axis('off')

axes[1].imshow(image_moving, cmap='gray')
axes[1].set_title("warped iamge")
axes[1].axis('off')


In [ ]:
angle, shift, aligned = estimate_rotation_translation_fourier(image_reference, image_moving)
print(f"Rotation: {angle:.2f}°")
print(f"Translation: {shift}")

# Create 1 row, 2 columns
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot first image
axes[0].imshow(image_reference, cmap='gray')
axes[0].set_title("fix image")
axes[0].axis('off')

axes[1].imshow(aligned, cmap='gray')
axes[1].set_title("warped iamge")
axes[1].axis('off')

In [ ]:
newczi = Path("test06.czi")

with pyczi.create_czi(newczi, exist_ok=True) as czidoc_w:
    # Loop over Z-planes and channels
    plane = {'C': 0, 'Z': 0, 'T': 0}
    czidoc_w.write(data=image_reference[..., np.newaxis], plane=plane)
    plane = {'C': 0, 'Z': 1, 'T': 0}
    czidoc_w.write(data=aligned[..., np.newaxis], plane=plane)
    plane = {'C': 0, 'Z': 2, 'T': 0}
    czidoc_w.write(data=image_moving[..., np.newaxis], plane=plane)

In [ ]:
def estimate_rotation_translation_fourier(reference_img, moving_img, upsample_factor=10):
    assert reference_img.shape == moving_img.shape, "Images must be the same shape"

    # Store original dtype
    orig_dtype = moving_img.dtype

    # Step 1: FFT magnitude
    f_ref = fftshift(fft2(reference_img))
    f_mov = fftshift(fft2(moving_img))
    mag_ref = np.abs(f_ref)
    mag_mov = np.abs(f_mov)

    # Step 2: Polar transform (linear for rotation only)
    center = np.array(reference_img.shape) / 2
    polar_ref = warp_polar(mag_ref, center=center, scaling='linear', preserve_range=True)
    polar_mov = warp_polar(mag_mov, center=center, scaling='linear', preserve_range=True)

    # Step 3: Estimate rotation angle
    shift, _, _ = phase_cross_correlation(polar_ref, polar_mov, upsample_factor=upsample_factor)
    rotation_deg = -shift[0] * 360 / polar_ref.shape[0]

    # Step 4: Apply rotation (preserve original data range)
    rotated = rotate(moving_img, rotation_deg, resize=False, order=2, mode='constant', cval=0.0, preserve_range=True)

    # Step 5: Estimate translation
    translation_shift, _, _ = phase_cross_correlation(reference_img, rotated, upsample_factor=upsample_factor)

    # Step 6: Apply translation (preserve range by keeping dtype conversion to the end)
    aligned = nd_shift(rotated, shift=translation_shift, order=2, mode='constant', cval=0.0)

    # Step 7: Cast back to original dtype
    aligned_final = aligned.astype(orig_dtype)

    return rotation_deg, translation_shift, aligned_final

def find_largest_centered_rectangle(mask):
    """
    Finds the largest rectangle centered at the image center
    that is fully contained in a binary mask.
    """
    h, w = mask.shape
    cy, cx = h // 2, w // 2

    max_up = max_down = max_left = max_right = 0

    # Vertical extent
    for i in range(cy, -1, -1):
        if mask[i, cx]:
            max_up += 1
        else:
            break
    for i in range(cy + 1, h):
        if mask[i, cx]:
            max_down += 1
        else:
            break

    # Horizontal extent
    for j in range(cx, -1, -1):
        if mask[cy, j]:
            max_left += 1
        else:
            break
    for j in range(cx + 1, w):
        if mask[cy, j]:
            max_right += 1
        else:
            break

    top = cy - max_up + 1
    bottom = cy + max_down
    left = cx - max_left + 1
    right = cx + max_right

    # Now shrink until the whole region is valid
    while True:
        region = mask[top:bottom, left:right]
        if region.shape[0] == 0 or region.shape[1] == 0:
            raise ValueError("No valid center-aligned rectangle found.")
        if np.all(region):
            break
        # shrink evenly
        if bottom - top > 1:
            top += 1
            bottom -= 1
        if right - left > 1:
            left += 1
            right -= 1

    return top, bottom, left, right

def crop_to_centered_valid_rectangle(img1, img2):
    shape = np.minimum(img1.shape, img2.shape)
    y, x = shape

    # Compute masks
    mask1 = np.logical_not(mask_low_and_saturation(img1)[:y, :x])
    mask2 = np.logical_not(mask_low_and_saturation(img2)[:y, :x])

    # Combine masks
    common_mask = mask1 & mask2

    # Find largest internal rectangle centered on center
    top, bottom, left, right = find_largest_centered_rectangle(common_mask)

    # Compute centers of original images
    center1 = np.array(img1.shape) // 2
    center2 = np.array(img2.shape) // 2

    # Calculate slice boundaries for each image
    crop1_row_start = center1[0] - (y // 2 - top)
    crop1_row_end   = center1[0] + (bottom - y // 2)
    crop1_col_start = center1[1] - (x // 2 - left)
    crop1_col_end   = center1[1] + (right - x // 2)

    crop2_row_start = center2[0] - (y // 2 - top)
    crop2_row_end   = center2[0] + (bottom - y // 2)
    crop2_col_start = center2[1] - (x // 2 - left)
    crop2_col_end   = center2[1] + (right - x // 2)

    # Extract crops
    crop1 = img1[crop1_row_start:crop1_row_end, crop1_col_start:crop1_col_end]
    crop2 = img2[crop2_row_start:crop2_row_end, crop2_col_start:crop2_col_end]

    # Compute origins in original image coordinates
    origin1 = np.array([crop1_row_start, crop1_col_start])
    origin2 = np.array([crop2_row_start, crop2_col_start])

    return crop1, crop2, origin1, origin2

def split_into_quadrants(img):
    h, w = img.shape
    mid_h, mid_w = h // 2, w // 2
    return [
        img[:mid_h, :mid_w],     # Top-left
        img[:mid_h, mid_w:],     # Top-right
        img[mid_h:, :mid_w],     # Bottom-left
        img[mid_h:, mid_w:]      # Bottom-right
    ]


# 0. Take original images, and then crop them using the centre as refence, this is to remove
# 0 values that can become an issue later on.
image_reference, image_moving, origin_reference, origin_moving = crop_to_centered_valid_rectangle(ref_img_ds, mov_img_ds)

# 1. Split image into quadrants
ref_split = split_into_quadrants(image_reference)
mov_split = split_into_quadrants(image_moving)

h, w = image_reference.shape
mid_h, mid_w = h // 2, w // 2

quad_centers = [
    (mid_h // 2, mid_w // 2),                  # Top-left
    (mid_h // 2, mid_w + mid_w // 2),          # Top-right
    (mid_h + mid_h // 2, mid_w // 2),          # Bottom-left
    (mid_h + mid_h // 2, mid_w + mid_w // 2)   # Bottom-right
]

# 2. Estimate transform for each quadrant
matched_ref = []
matched_mov = []

for ref_patch, mov_patch, center in zip(ref_split, mov_split, quad_centers):
    try:
        angle, shift, _ = estimate_rotation_translation_fourier(ref_patch, mov_patch)
        moved_center = np.array(center) + np.array(shift)
        #TODO: NEED to be roated, if not it is not completely correct!!!!
        matched_ref.append(center)
        matched_mov.append(moved_center)
    except Exception as e:
        print(f"Skipping one quadrant due to error: {e}")
        continue

# 3. Estimate global rigid transform
if len(matched_ref) < 2:
    raise RuntimeError("Not enough matching points to estimate a transform.")

tform = estimate_transform('euclidean', src=np.array(matched_mov), dst=np.array(matched_ref))

# 4. Apply global transform
aligned_crop = warp(image_moving, inverse_map=tform.inverse, preserve_range=True, mode='constant', cval=0.0)
aligned_crop = aligned_crop.astype(image_moving.dtype)

# calculate back to original frame of reference
matched_ref_global = [np.array(p) + origin_reference for p in matched_ref]
matched_mov_global = [np.array(p) + origin_moving for p in matched_mov]
tform = estimate_transform('euclidean', src=np.array(matched_mov_global), dst=np.array(matched_ref_global))

# 4. Apply global transform to original images
aligned = warp(mov_img_ds, inverse_map=tform.inverse, preserve_range=True, mode='constant', cval=0.0)
aligned = aligned.astype(mov_img_ds.dtype)

In [ ]:
# Create 1 row, 2 columns
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot first image
axes[0].imshow(image_reference, cmap='gray')
axes[0].set_title("fix image")
axes[0].axis('off')

axes[1].imshow(aligned, cmap='gray')
axes[1].set_title("warped iamge")
axes[1].axis('off')

In [ ]:
newczi = Path("test07.czi")

with pyczi.create_czi(newczi, exist_ok=True) as czidoc_w:
    # Loop over Z-planes and channels
    plane = {'C': 0, 'Z': 0, 'T': 0}
    czidoc_w.write(data=image_reference[..., np.newaxis], plane=plane)
    plane = {'C': 0, 'Z': 1, 'T': 0}
    czidoc_w.write(data=aligned_crop[..., np.newaxis], plane=plane)
    plane = {'C': 0, 'Z': 2, 'T': 0}
    czidoc_w.write(data=image_moving[..., np.newaxis], plane=plane)

In [ ]:
newczi = Path("test08.czi")

with pyczi.create_czi(newczi, exist_ok=True) as czidoc_w:
    # Loop over Z-planes and channels
    plane = {'C': 0, 'Z': 0, 'T': 0}
    czidoc_w.write(data=ref_img_ds[..., np.newaxis], plane=plane)
    plane = {'C': 0, 'Z': 1, 'T': 0}
    czidoc_w.write(data=aligned[..., np.newaxis], plane=plane)
    plane = {'C': 0, 'Z': 2, 'T': 0}
    czidoc_w.write(data=mov_img_ds[..., np.newaxis], plane=plane)